----------------------------------------------------------------------------------------------------

# ***Testando APIs RESTful Assíncronas em FastAPI***

----------------------------------------------------------------------------------------------------

# ***1. Reestruturando o projeto para realização de testes*** 

## ***1.1. Criação do arquivo 'conftest.py' na pasta 'tests'***

In [ ]:
import asyncio
import os

import pytest_asyncio
from httpx import ASGITransport, AsyncClient

os.environ.setdefault("DATABASE_URL", f"sqlite:///tests.db")


@pytest_asyncio.fixture
async def db(request):
    from src.app.database import database, engine, metadata
    from src.models.post import posts

    await database.connect()
    metadata.create_all(engine)

    def teardown():
        async def _teardown():
            await database.disconnect()
            metadata.drop_all(engine)

        asyncio.run(_teardown())

    request.addfinalizer(teardown)


## ***1.2. Atualização do arquivo 'database.py' na pasta 'src'***

In [ ]:
import os

import databases 
import sqlalchemy 

DATABASE_URL = os.getenv("DATABASE_URL",'sqlite:///./blog.db')

# cria database
database = databases.Database(DATABASE_URL)
# instanciando a classe SQLalchemmy
metadata = sqlalchemy.MetaData()
engine = sqlalchemy.create_engine(DATABASE_URL, connect_args = {"check_same_thread": False})

## ***1.3. Atualização do arquivo 'conftest.py' na pasta 'tests'***

In [ ]:
import asyncio
import os

import pytest_asyncio
from httpx import ASGITransport, AsyncClient

os.environ.setdefault("DATABASE_URL", f"sqlite:///tests.db")


# Primeira função - Retorna o banco de dados
@pytest_asyncio.fixture
async def db(request):
    from src.app.database import database, engine, metadata
    from src.models.post import posts

    await database.connect()
    metadata.create_all(engine)

    def teardown():
        async def _teardown():
            await database.disconnect()
            metadata.drop_all(engine)

        asyncio.run(_teardown())

    request.addfinalizer(teardown)


# Segunda função - Retorna o http cliente
@pytest_asyncio.fixture
async def client(db):
    from src.app.main import app

    transport = ASGITransport(app = app)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    async with AsyncClient(base_url = "http://test", transport= transport, headers= headers) as client:
        yield client



# Terceira função - Facilita o acess_token

## ***1.4. Atualização do arquivo 'conftest.py' na pasta 'tests' com a nova função***

In [ ]:
import asyncio
import os

import pytest_asyncio
from httpx import ASGITransport, AsyncClient

os.environ.setdefault("DATABASE_URL", f"sqlite:///tests.db")


# Primeira função - Retorna o banco de dados
@pytest_asyncio.fixture
async def db(request):
    from src.app.database import database, engine, metadata
    from src.models.post import posts

    await database.connect()
    metadata.create_all(engine)

    def teardown():
        async def _teardown():
            await database.disconnect()
            metadata.drop_all(engine)

        asyncio.run(_teardown())

    request.addfinalizer(teardown)


# Segunda função - Retorna o http client
@pytest_asyncio.fixture
async def client(db):
    from src.app.main import app

    transport = ASGITransport(app = app)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    async with AsyncClient(base_url = "http://test", transport= transport, headers= headers) as client:
        yield client



# Terceira função - Facilita o acess_token
async def acess_token(client: AsyncClient):
    response = await client.post("/auth/login", json={"user_id": 1})
    return response.json()["acess_token"]

----------------------------------------------------------------------------------------------------

# ***2. Definindo o respose para cada vez que o login for efetuado*** 

## ***2.1. Atualização do arquivo 'test_login.py' na pasta 'auth' em 'controllers' da pasta 'integration'***

In [ ]:
from fastapi import status
from httpx import AsyncClient

async def test_login_sucess(client: AsyncClient):
    # Given
    data = {"user_id": 1}

    # When
    response = await client.post("/auth/login", json= data)

    # Then
    assert response.status_code == status.HTTP_200_OK
    assert response.json()["access_token"] is not None

Esses três comentários representam o padrão GWT (Given / When / Then), muito usado em testes. 

É uma forma de organizar o raciocínio do teste de maneira clara e legível:

----------------------------------------------------------------------------------------------------

# ***3. Definindo cada processo de execução teste para o CRUD*** 

Precisamos definir quais os responses para cada situação ao serem executadas.

## ***3.1. CREATE - Criação do arquivo 'test_create_post.py' na pasta 'post' em 'controllers' da pasta 'integration'***

In [ ]:
from fastapi import status
from httpx import AsyncClient


async def test_create_post_success(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}
    data = {"title": "post 1", "content": "some content", "published_at": "2026-05-11", "published": True}

    # When
    response = await client.post("/posts/", json= data, headers= headers)

    # Then
    content = response.json()

    assert response.status_code == status.HTTP_201_CREATED
    assert content["id"] is not None


async def test_create_post_invalid_payload_fail(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}
    data = {"content": "some content", "published_at": "2026-05-11", "published": True}

    # When
    response = await client.post("/posts/", json= data, headers= headers)

    # Then
    content = response.json()


    assert response.status_code == status.HTTP_422_UNPROCESSABLE_CONTENT
    assert content["detail"][0]["loc"] == ["body", "title"]


async def test_create_post_not_authenticated_fail(client: AsyncClient):
    # Given
    data = {"content": "some content", "published_at": "2026-05-11", "published": True}

    # When
    response = await client.post("/post/", json = data, headers={})

    # Then
    assert response.status_code == status.HTTP_401_UNAUTHORIZED

## ***3.2. READ ALL - Criação do arquivo 'test_read_all.py' na pasta 'post' em 'controllers' da pasta 'integration'***

In [ ]:
import pytest
import pytest_asyncio
from fastapi import status
from httpx import AsyncClient

@pytest_asyncio.fixture(autouse= True) # 'autouse' = fará que todos os métodos utilizem o populate_post
async def populate_posts(db):
    from src.schemas.post import PostIn
    from src.services.post import PostService

    service = PostService()

    await service.create(PostIn(title= "post 1", content= "some content", published = True))
    await service.create(PostIn(title= "post 2", content= "some content", published = True))
    await service.create(PostIn(title= "post 3", content= "some content", published = False))


@pytest.mark.parametrize("published, total", [("on", 2), ("off", 1)])
async def test_read_posts_by_status_success(client: AsyncClient, access_token: str, published: str, total: int):
    # Given
    params = {"published": published, "limit": 10}
    headers = {"Authorization": f"Bearer {access_token}"}

    # When
    response = await client.get("/posts/", params= params, headers= headers)

    # Then
    content = response.json()

    assert response.status_code == status.HTTP_200_OK
    assert len(content) == total


async def test_read_posts_limit_success(client: AsyncClient, access_token: str):
    #Given
    params = {"published": "on", "limit": 1}
    headers = {"Authorization": f"Bearer {access_token}"}

    #When
    response = await client.get("/posts/", params= params, headers= headers)

    #Then
    content = response.json()

    assert response.status_code == status.HTTP_200_OK
    assert len(content) == 1


async def test_read_posts_not_authenticated_fail(client: AsyncClient):
    # Given
    params = {"published": "on", "limit": 1}

    # When
    response = await client.get("/posts/", params= params, headers= {})

    # Then
    assert response.status_code == status.HTTP_401_UNAUTHORIZED


async def test_read_posts_empty_parameters_fail(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}

    # When
    response = await client.get("/posts/", params= {}, headers= headers)

    # Then
    assert response.status_code == status.HTTP_422_UNPROCESSABLE_CONTENT

## ***3.3. READ ONE - Criação do arquivo 'test_read_post.py' na pasta 'post' em 'controllers' da pasta 'integration'***

In [ ]:
import pytest_asyncio
from fastapi import status
from httpx import AsyncClient

@pytest_asyncio.fixture(autouse= True)
async def populate_posts(db):
    from src.schemas.post import PostIn
    from src.services.post import PostService

    service = PostService()

    await service.create(PostIn(title= "post 1", content= "some content", published = True))
    await service.create(PostIn(title= "post 2", content= "some content", published = True))
    await service.create(PostIn(title= "post 3", content= "some content", published = False))


async def test_read_posts_success(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}
    post_id = 1

    # When
    response = await client.get("/posts/{post_id}", headers= headers)

    # Then
    content = response.json()

    assert response.status_code == status.HTTP_200_OK
    assert content["id"] == post_id


async def test_read_posts_not_authenticated_fail(client: AsyncClient):
    # Given
    post_id = 1

    # When
    response = await client.get("/posts/{post_id}", headers= {})

    # Then
    assert response.status_code == status.HTTP_401_UNAUTHORIZED


async def test_read_posts_not_found_fail(client: AsyncClient, access_token: str):
    # Given
    headers = {"Authorization": f"Bearer {access_token}"}
    post_id = 4

    # When
    response = await client.get("/posts/{post_id}", headers= headers)

    # Then
    content = response.json()

    assert response.status_code == status.HTTP_404_NOT_FOUND

## ***3.4. UPDATE - Criação do arquivo 'test_update_post.py' na pasta 'post' em 'controllers' da pasta 'integration'***